# AII Time-Series

AI Intensity Index (AII) — quarterly scalar measuring weighted density of AI-related language
in Workday canonical_v2 SEC filings.

Run `python3 -m measures.run_aii` first to generate `data/processed/aii_quarterly.csv`.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

DATA_DIR  = Path("../data/processed")
PLOTS_DIR = DATA_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_DIR / "aii_quarterly.csv")
df = df.sort_values(["year", "quarter"]).reset_index(drop=True)

print(f"{len(df)} quarters loaded ({df['period'].iloc[0]} – {df['period'].iloc[-1]})")
df[["period", "doc_count", "aii", "aii_delta", "bucket_generative_ai"]].head(10)

53 quarters loaded (2012-Q4 – 2025-Q4)


,period,doc_count,aii,aii_delta,bucket_generative_ai
0,2012-Q4,1,0.000000,NaN,0
1,2013-Q1,1,0.000000,0.000000,0
2,2013-Q2,1,0.000000,0.000000,0
3,2013-Q3,1,0.000000,0.000000,0
4,2013-Q4,1,0.000000,0.000000,0
5,2014-Q1,1,0.000000,0.000000,0
6,2014-Q2,1,0.045127,0.045127,0
7,2014-Q3,1,0.042627,-0.002500,0
8,2014-Q4,1,0.042994,0.000367,0
9,2015-Q1,1,0.057484,0.014490,0


In [2]:
# ── Primary AII time-series ──────────────────────────────────────────────────
# Two-panel: top = AII line + area fill, bottom = QoQ delta bar

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(14, 8),
    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08},
    sharex=True,
)

x      = range(len(df))
aii    = df["aii"].values
delta  = df["aii_delta"].values
labels = df["period"].values

# Shade pre / post ChatGPT launch (2022-Q4)
gpt_idx = df.index[df["period"] == "2022-Q4"].tolist()
gpt_x   = gpt_idx[0] if gpt_idx else len(df)

ax_top.axvspan(0, gpt_x, alpha=0.04, color="steelblue", label="Pre-GenAI era")
ax_top.axvspan(gpt_x, len(df), alpha=0.07, color="darkorange", label="Post-GenAI era")

ax_top.fill_between(x, aii, alpha=0.15, color="steelblue")
ax_top.plot(x, aii, color="steelblue", linewidth=2, marker="o", markersize=4, label="AII")

ax_top.set_ylabel("AII (weighted density × 1000)", fontsize=11)
ax_top.set_title("Workday AI Intensity Index — Quarterly (canonical_v2)", fontsize=13)
ax_top.legend(fontsize=9)
ax_top.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))

# Delta bar
colors = ["green" if (v is not None and v >= 0) else "red" for v in delta]
ax_bot.bar(x, [v if pd.notna(v) else 0 for v in delta], color=colors, alpha=0.7)
ax_bot.axhline(0, color="black", linewidth=0.8)
ax_bot.set_ylabel("QoQ Δ", fontsize=10)

# X-axis: show every 4th label to avoid clutter
step = max(1, len(df) // 12)
ax_bot.set_xticks([i for i in x if i % step == 0])
ax_bot.set_xticklabels(
    [labels[i] for i in x if i % step == 0],
    rotation=45, ha="right", fontsize=8,
)

out = PLOTS_DIR / "aii_quarterly.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")

Saved: ../data/processed/plots/aii_quarterly.png


In [3]:
# ── Bucket decomposition ─────────────────────────────────────────────────────
# Stacked bar showing weighted contribution of each bucket per quarter

from measures.aii import AII_TERM_BUCKETS

weights = {b: AII_TERM_BUCKETS[b]["weight"] for b in AII_TERM_BUCKETS}

# Weighted raw counts per quarter (not yet normalized by tokens)
classic  = df["bucket_classic_ai"] * weights["classic_ai"]
genai    = df["bucket_generative_ai"] * weights["generative_ai"]
adjacent = df["bucket_adjacent_automation"] * weights["adjacent_automation"]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x, classic,  label=f"Classic AI (×{weights['classic_ai']})",          color="steelblue",   alpha=0.85)
ax.bar(x, genai,    label=f"Generative AI (×{weights['generative_ai']})",     color="darkorange",  alpha=0.85, bottom=classic)
ax.bar(x, adjacent, label=f"Adjacent Automation (×{weights['adjacent_automation']})", color="mediumseagreen", alpha=0.85,
       bottom=classic + genai)

ax.set_xticks([i for i in x if i % step == 0])
ax.set_xticklabels(
    [labels[i] for i in x if i % step == 0],
    rotation=45, ha="right", fontsize=8,
)
ax.set_ylabel("Weighted term counts", fontsize=11)
ax.set_title("AII Bucket Decomposition — Weighted Term Counts per Quarter", fontsize=13)
ax.legend(fontsize=9)

out = PLOTS_DIR / "aii_bucket_decomposition.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")

Saved: ../data/processed/plots/aii_bucket_decomposition.png


In [4]:
# ── Sanity checks ────────────────────────────────────────────────────────────

print("=" * 50)
print("AII Sanity Checks")
print("=" * 50)

nonzero = df[df["aii"] > 0]
print(f"First non-zero AII quarter : {nonzero['period'].iloc[0] if len(nonzero) else 'none'}")
print(f"Max AII quarter            : {df.loc[df['aii'].idxmax(), 'period']} ({df['aii'].max():.3f})")
print(f"Max delta quarter          : {df.loc[df['aii_delta'].idxmax(), 'period']} ({df['aii_delta'].max():+.3f})")

genai_quarters = df[df["bucket_generative_ai"] > 0]
print(f"First GenAI hits quarter   : {genai_quarters['period'].iloc[0] if len(genai_quarters) else 'none'}")
print(f"Quarters with GenAI > 0    : {len(genai_quarters)}")

assert df["aii_delta"].isna().sum() == 1, "Expected exactly 1 null delta (first quarter)"
assert (df["aii"] >= 0).all(), "All AII values must be non-negative"
assert (df["quarter_tokens"] > 0).all(), "No zero-token quarters"
print("\nAll assertions passed.")

AII Sanity Checks
First non-zero AII quarter : 2014-Q2
Max AII quarter            : 2020-Q1 (0.262)
Max delta quarter          : 2019-Q4 (+0.132)
First GenAI hits quarter   : 2023-Q2
Quarters with GenAI > 0    : 11

All assertions passed.


In [5]:
# ── Cross-signal correlation ──────────────────────────────────────────────────
# AII vs. existing ai_intensity from quarterly_signals.csv

qs_path = DATA_DIR / "quarterly_signals.csv"
if qs_path.exists():
    qs = pd.read_csv(qs_path)
    merged = df.merge(qs[["period", "ai_intensity"]], on="period", how="left")
    corr = merged[["aii", "ai_intensity"]].corr().iloc[0, 1]
    print(f"AII vs. ai_intensity Pearson r = {corr:.3f}")
    print("(Expected: positive, not 1.0 — different denominators and weighting)")
else:
    print("quarterly_signals.csv not found; skipping correlation check.")

AII vs. ai_intensity Pearson r = 0.584
(Expected: positive, not 1.0 — different denominators and weighting)
